# Bonus B5 solution, notebook 2: reject a refund, and ask twice for the same invoice

The worked answers to the two challenge tasks of `b5b_agent_framework_at_the_desk.ipynb`. The setup repeats the
notebook's lookups, agent and refund tool, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import AsyncOpenAI
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

client = OpenAIChatClient(model=MODEL, async_client=AsyncOpenAI(timeout=60, max_retries=2))
OPTIONS = {"reasoning": {"effort": EFFORT}}

The three lookups, each on a connection of its own, and the function that lists a run's tool calls.

In [3]:
import sqlite3
from contextlib import closing

DB_URI = f"file:{ROOT / 'data' / 'chinook.db'}?mode=ro"


def query(sql, args):
    """Run one read-only query on a connection of its own, and return every row."""
    with closing(sqlite3.connect(DB_URI, uri=True)) as con:
        return con.execute(sql, args).fetchall()


def find_customer(name: str) -> str:
    """Find a shop customer by full name. Returns the matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = query("SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
                 "WHERE FirstName || ' ' || LastName = ?", (name,))
    if not rows:
        return f"No match for a customer named {name}."
    return str([{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows])


def get_invoices(customer_id: int) -> str:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = query("SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,))[0]
    if not count:
        return f"No match for invoices of customer {customer_id}."
    return str({"customer_id": customer_id, "invoice_count": count, "total": total})


def get_support_rep(customer_id: int) -> str:
    """Name the support rep who looks after a customer."""
    rows = query("SELECT e.FirstName || ' ' || e.LastName FROM customers c "
                 "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,))
    if not rows:
        return f"No match for customer {customer_id}."
    return str({"customer_id": customer_id, "support_rep": rows[0][0]})

In [4]:
def tool_calls(answer):
    """The tools a run asked for, in order, with their arguments."""
    return [f"{content.name}({content.arguments})"
            for message in answer.messages
            for content in message.contents
            if content.type == "function_call"]

The refund workflow is imported from `refund_workflow.py` in the lab folder. This notebook sits in `solved/`, one
folder below it, so we add the lab folder to `sys.path` before the import. Then the folder the workflow writes to is
emptied, and the refund tool and the desk agent are built as in Part D.

In [5]:
import shutil
import subprocess
import sys

LAB = ROOT / "bonus" / "B5_Microsoft_Agent_Framework"
sys.path.insert(0, str(LAB))

from refund_workflow import FOLDER, build_workflow, refund_rows

shutil.rmtree(FOLDER, ignore_errors=True)

In [6]:
# invoice number -> the id of the checkpoint where its review is waiting
WAITING = {}


async def request_refund(invoice_id: int) -> str:
    """Ask for a refund of one invoice. A reviewer has to approve it before anything is written."""
    workflow, storage = build_workflow()
    run = await workflow.run(invoice_id)

    # The workflow ended at once: there was no such invoice
    if run.get_outputs():
        return run.get_outputs()[0]

    # The workflow is waiting for the reviewer
    refund = run.get_request_info_events()[0].data
    WAITING[invoice_id] = workflow.get_last_checkpoint_id()
    return (f"A refund of ${refund['amount']} for invoice {invoice_id} is waiting for a reviewer. "
            "Nothing has been refunded.")

In [7]:
DESK_RULES = (
    "You work at the support desk of the Chinook music shop. Use the tools for every fact about a customer: "
    "find the customer by full name first, then look up what was asked. Answer in one or two sentences."
)

desk_with_refunds = Agent(
    client=client,
    name="Desk",
    instructions=DESK_RULES + " Use request_refund when the rep asks for a refund.",
    tools=[find_customer, get_invoices, get_support_rep, request_refund],
    default_options=OPTIONS,
)


def answer_review(invoice_id, decision):
    # Start resume_refund.py as a new process and print what it printed
    done = subprocess.run(
        [sys.executable, str(LAB / "resume_refund.py"), WAITING[invoice_id], decision],
        capture_output=True, text=True,
    )
    print(done.stdout or done.stderr)

---

## Task 1: Reject a refund

The rep asks for a refund of invoice 229.

In [8]:
answer = await desk_with_refunds.run("Puja Srivastava asks for a refund of invoice 229. Please file it.")

print(answer.text)
print()
for call in tool_calls(answer):
    print(call)
print()
print("refunds table:", refund_rows())

I filed the refund request for invoice 229. The $13.86 refund is awaiting reviewer approval; nothing has been refunded yet.

find_customer({"name":"Puja Srivastava"})
request_refund({"invoice_id":229})

refunds table: []


The agent called `request_refund` with invoice 229 and reports that the refund is waiting for a reviewer. The
reviewer answers `reject`, from a new process.

In [9]:
answer_review(229, "reject")
print("refunds table:", refund_rows())

process id: 3382
reviews waiting: 1
the question: {'invoice_id': 229, 'customer_id': 59, 'amount': 13.86}
answer: reject
workflow output: ['Refund for invoice 229 rejected. Nothing written.']
refunds table: []

refunds table: []


The new process found one waiting review: invoice 229, customer 59, \$13.86, the invoice's total from the shop's
database. After `reject`, the `review` step ended the workflow with `Refund for invoice 229 rejected. Nothing
written.` It sent nothing along the edge to `file_refund`, so the refunds table has no row for 229.

---

## Task 2: Ask twice for the same invoice

First the refund of invoice 284 goes through once: the rep asks, the reviewer approves.

In [10]:
answer = await desk_with_refunds.run("Puja Srivastava asks for a refund of invoice 284. Please file it.")
print(answer.text)
print()

answer_review(284, "approve")

I submitted the refund request for invoice 284. The $8.91 refund is awaiting reviewer approval; nothing has been refunded yet.



process id: 3386
reviews waiting: 1
the question: {'invoice_id': 284, 'customer_id': 59, 'amount': 8.91}
answer: approve
workflow output: ['Refund for invoice 284 written.']
refunds table: [(284, 59, 8.91)]



One row for invoice 284. Now another rep asks for the same refund.

In [11]:
answer = await desk_with_refunds.run("Please refund invoice 284 for Puja Srivastava.")

print(answer.text)
print()
for call in tool_calls(answer):
    print(call)

Your refund request for $8.91 on invoice 284 has been submitted and is awaiting reviewer approval. No refund has been issued yet.

find_customer({"name":"Puja Srivastava"})
request_refund({"invoice_id":284})


The agent started the workflow again, and a second review is waiting. Nothing in `check_invoice` or `review` looks
at the refunds table, so the request travels the same path as the first time. The reviewer approves again.

In [12]:
answer_review(284, "approve")
print("refunds table:", refund_rows())

process id: 3387
reviews waiting: 1
the question: {'invoice_id': 284, 'customer_id': 59, 'amount': 8.91}
answer: approve
workflow output: ['Refund for invoice 284 is already on file. Nothing written.']
refunds table: [(284, 59, 8.91)]

refunds table: [(284, 59, 8.91)]


The output is `Refund for invoice 284 is already on file. Nothing written.`, and the table still has one row for
invoice 284. The key on the table held, as it did for the repeated answer in Part E.

The cost was a reviewer's time: a person was asked to approve a refund that could not be written.

**Which step to change.** `check_invoice`, the first step. It already ends the workflow in words when the invoice
does not exist. It could also read the `refunds` table for the invoice number, and end the workflow with "Invoice
284 is already refunded" when a row is there. The agent would pass that sentence to the rep, and no review would
wait. The key on the table stays: the early check saves the reviewer's time, and the key is what makes a second
write impossible.

Back to [the notebook](../b5b_agent_framework_at_the_desk.ipynb).